# WS5 — Valuing the Setup Pitch: A Tabular MDP with an Honest Sequential Gate

**Workstream 5 of the pitch-sequencing rigor ladder — the first *sequential* prescriptive rung (Phase B).**
This notebook builds a small **tabular Markov decision process** over the count and a sliver of ordered
history, plans it with undiscounted **policy iteration**, and evaluates the softened policy **two
independent ways** — a transparent **model-based** value from the estimated MDP and an **off-policy**
value on held-out logged rows through `eval/ope`. It is the first rung that can value a **setup pitch**:
an action whose payoff is not this pitch's reward but the *state it creates for the next pitch*.

## The rung that can finally cash a setup

WS4 measured a **ceiling**. Its myopic bandit could reach only the *family-differential* sliver of the
planted effect (`~0.003` run of a `~0.032` whiff boost) because the boost is a **state-value** effect —
fixed by the two prior pitches, owed equally to every current family, invisible to a one-step value. WS4
handed the ladder a falsifiable target (decision **D40**): *beat `0.003` by valuing the setup a bandit
provably cannot.* WS5 is the first model that can hold that target, because its MDP keeps the term a
bandit drops — `E[v(s') | s, a]`, today's action shifting *tomorrow's* state value. The state ladder is
a **velocity-gap trigger flag**: choosing a current family whose velo band differs enough from the
previous pitch's drives the *next* state's trigger to `1`, and a triggered state carries the higher
reward. Policy iteration therefore values actions for the trigger they *create*.

## The chapter's twin payloads

1. **Representability, proven.** A constructed-world unit test shows the trigger design's optimal policy
   *takes the setup pitch* and its value is provably higher than the count design's — the machinery
   **can** cash a setup (§4).
2. **Certification, variance-bound.** On the full positive fixture the held-out trigger-count gap is
   directionally positive in all three lenses, but its refit-bootstrap confidence interval does not
   clear the `+0.003` ceiling at synthetic scale — an honest **`SETUP_INCONCLUSIVE`** (decision **D39**).
   The binding constraint is not the point estimate; it is the **sequential-OPE variance** (decision
   **D44**), and that is a *data-scale* question Phase 2 answers (§6, §7).

WS5 is also the study's **OPE cross-validator** (decision **D42**): the same softened policy scored by
the estimated MDP, by step-wise DR, and by FQE, with their agreement judged on real CIs — divergence
**reported, never silently averaged** (§5).

## The three findings this notebook keeps separate (SPEC §0, verbatim)

> 1. **Selection structure** — prior pitches help predict *what is thrown next*.
> 2. **Predictive sequencing value** — prior pitches help predict the *outcome* of the current pitch,
>    after conditioning on the current pitch and game state.
> 3. **Prescriptive/causal value** — *changing* the sequence would improve outcomes.

WS5 lives at the hardest edge of finding #3: it tests whether *acting* on the estimated MDP beats the
observed policy **within support**, and reports `SETUP_INCONCLUSIVE` when the held-out variance cannot
resolve the gap. It does not *assume* the trigger is causal; it tests one *representable* mechanism.

## The DATA_MODE toggle

This notebook is a **scaffold**. Phase 2 runs it on the real Statcast decision table; here a single
toggle, `DATA_MODE`, selects the world:

- `'synth_positive'` — the oracle's **positive world** (a planted velocity-transition whiff boost).
  **Default**, because it exercises the full setup story end-to-end.
- `'synth_null'` — the oracle's **null world** (no ordered *outcome* effect); the trigger flag is inert
  and the verdict is `SEQ_NEUTRAL_MDP`.
- `'real'` — the real decision table (RUNBOOK Steps 1, WS5.1); pass `--ws3-dir` to reuse WS3's behavior
  propensities, else the state-conditional empirical behavior is fit from train counts.

## How to read this notebook

Every code step is bracketed by plain-worded markdown: **before** each cell we say what will happen and
why; **after** each cell we say how to read what came out. Numbers that depend on the real data are
written as `{PLACEHOLDER}` in the companion `PAPER.md`; here they simply appear when you run the cell.
The **Results** section (§8) is *branched* on three axes — **design-ladder** (S+/S0/S−), **D42
agreement** (A+/A−), and **verdict** (`SETUP_EXPLOITED` / `SETUP_INCONCLUSIVE` / `SEQ_NEUTRAL_MDP`) — a
code cell inspects the computed report and prints which branch fired; the markdown that follows holds
the pre-written interpretation for every branch. The exact formulas live in `THEORY.md`; the exact code
in `workstreams/ws5_tabular_mdp/model.py` and `run_ws5.py`; the plain-English tour in `SEAN-README.md`.

> **Scale note.** Tabular counting and policy iteration are near-instant (the state spaces are `≤ 220`
> states). The cost is the OPE pass — step-wise DR and FQE over the held-out rows — and, above all, the
> **FQE refit cluster bootstrap** (§6), which refits FQE once per design × α per replicate. The
> **committed-validation** numbers quoted in the markdown come from the full run (`n_games = 2000`,
> `--fqe-boot 200` ≈ 3,000 refits ≈ 57 min, peak `~1.2 GB`); the small in-notebook budget below
> reproduces the **verdicts** and the qualitative story, not the exact CI widths. On real data, lower
> `--fqe-boot` to 50–100 if the refit bootstrap is slow (it changes only the CI resolution).

## 1. Setup

We put the repository root on `sys.path`, load the shared study config (the seed and the `π_α` grid come
from it), pick the world with `DATA_MODE`, and set the three D43 state designs and the modest in-notebook
budgets. Nothing here is WS5-specific modelling — it is the shared plumbing every workstream opens with.

In [ ]:
import sys
import json
import platform
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# --- locate the repository root (works from repo root or from notebooks/) ---
REPO_ROOT = Path.cwd()
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "pyproject.toml").exists() and (_p / "workstreams").is_dir():
        REPO_ROOT = _p
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- shared foundation (WS0) ---
from pitchseq.config import load_config
from pitchseq.splits import make_splits, cluster_bootstrap_indices
from pitchseq.families import FAMILIES
from pitchseq.eval import ope

# --- WS5 (this workstream) ---
from workstreams.ws5_tabular_mdp.model import (
    STATE_DESIGNS, RICHEST_DESIGN, TERMINALS, PREV_NONE, N_COUNT, N_PREV, XX_INDEX,
    DEFAULT_VELO_GAP_THRESHOLD,
    encode_states, estimate_mdp, estimate_behavior_policy, policy_evaluation, policy_iteration,
    greedy_policy_matrix, soften_policy, simulate, setup_diagnostics, family_velo_bands,
    prev_and_trigger, onehot_tabular_regressor, policy_to_row_probs,
)
# The runnable pipeline + the D40 ceiling constant (no re-implementation of the pipeline here):
from workstreams.ws5_tabular_mdp.run_ws5 import run_ws5, _format_headline, D40_MYOPIC_CEILING

CONFIG = load_config()
SEED = int(CONFIG.get("seeds", {}).get("global", 20260713))

# The world this run analyses: 'synth_positive' | 'synth_null' | 'real'.
# Default 'synth_positive': it plants a velocity-transition whiff boost, so the setup story (a
# state-value effect a sequential planner CAN cash but a bandit cannot) is exercised end-to-end.
DATA_MODE = "synth_positive"

# The D43 state-space ladder (decision D43) -- the C / L1 / O-lite measurement views in STATE space:
#   count               ~ C      (balls, strikes) only                       -> 12 (+4 terminals) = 16
#   count_prev          ~ L1     x previous family (9, incl NONE) = 108       -> (+4) = 112
#   count_prev_trigger  ~ O-lite x a velo-gap trigger flag (2)   = 216        -> (+4) = 220
DESIGNS = list(STATE_DESIGNS)                       # ("count", "count_prev", "count_prev_trigger")
assert RICHEST_DESIGN == "count_prev_trigger"

# The pi_alpha grid (SPEC 9): 0 = behavior (the OPE baseline), 1 = the pure greedy target.
ALPHAS = [float(a) for a in CONFIG.get("ope", {}).get("conservative_alpha", [0.0, 0.1, 0.25, 0.5, 1.0])]

# The velo-gap trigger threshold (mph) -- echoes the planted positive-world mechanism.
THRESHOLD = DEFAULT_VELO_GAP_THRESHOLD               # 5.0

# --- in-notebook budgets (a quick scaffold pass; the committed run uses n_games=2000, fqe_boot=200) ---
NB_N_GAMES = 800          # synthetic-world size (ignored when DATA_MODE == 'real')
NB_FQE_BOOT = 30          # FQE REFIT-bootstrap replicates (the resolving CI; small here for speed)
NB_N_BOOT = 120           # step-wise-DR contribution-bootstrap replicates (CI width only)
NB_SIM_EPISODES = 1500    # simulator rollouts for the D42 model-based cross-check
WORLD_SEED = 7            # the seed run_ws5 uses to build a synthetic world (its default)
ALPHA_R = 12.0            # reward shrinkage (the synth-tuned value so the richer designs generalise)
FIDX = {f: i for i, f in enumerate(FAMILIES)}        # family -> action index

# Phase-2 real inputs (Step 1 builds the decision table; --ws3-dir is optional, D33).
REAL_TABLE_PATH = REPO_ROOT / "data" / "processed" / "decision_table.parquet"
WS3_DIR = REPO_ROOT / "results" / "ws3"

print(f"repo root : {REPO_ROOT}")
print(f"DATA_MODE : {DATA_MODE}")
print(f"designs   : {DESIGNS}   (count~C, count_prev~L1, count_prev_trigger~O-lite; D43)")
print(f"alphas    : {ALPHAS}   threshold(mph) : {THRESHOLD}   ceiling(D40) : {D40_MYOPIC_CEILING}")
print(f"budgets   : n_games={NB_N_GAMES}  fqe_boot={NB_FQE_BOOT}  n_boot={NB_N_BOOT}  sim={NB_SIM_EPISODES}")
print(f"seed      : {SEED}   world_seed : {WORLD_SEED}")

### Plotting style (fixed, colorblind-safe design colours)

We use the Okabe–Ito qualitative palette (colorblind-safe), **byte-identical to WS1/WS2/WS3/WS4**, and
map each design to the colour of the measurement view it mirrors — `count`→`C` (blue), `count_prev`→`L1`
(bluish-green), `count_prev_trigger`→`O` (vermillion) — so a figure here reads the same as the ablation
plots elsewhere. Spines are stripped to left+bottom and gridlines dropped; `REF_COLOR` (black) is the
neutral colour for reference lines (the D40 ceiling, the behavior baseline).

In [ ]:
# Okabe-Ito qualitative palette (colorblind-safe) -- identical to WS1/WS2/WS3/WS4.
OKABE_ITO = {
    "orange":         "#E69F00",
    "sky_blue":       "#56B4E9",
    "bluish_green":   "#009E73",
    "yellow":         "#F0E442",
    "blue":           "#0072B2",
    "vermillion":     "#D55E00",
    "reddish_purple": "#CC79A7",
    "black":          "#000000",
}
# Fixed design -> colour, keyed by the C / L1 / O view each design mirrors (identical to WS3/WS4).
DESIGN_COLORS = {
    "count":              OKABE_ITO["blue"],          # ~ C  (context/count only)
    "count_prev":         OKABE_ITO["bluish_green"],  # ~ L1 (previous pitch)
    "count_prev_trigger": OKABE_ITO["vermillion"],    # ~ O-lite (adds the velo-gap trigger flag)
}
DESIGN_LABEL = {"count": "count (~C)", "count_prev": "count_prev (~L1)",
                "count_prev_trigger": "count_prev_trigger (~O-lite)"}
REF_COLOR = OKABE_ITO["black"]  # neutral: the D40 ceiling and behavior baseline reference lines

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 11,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "figure.autolayout": True,
})


def style_axes(ax):
    """Left+bottom spines only; no top/right. Returns the axis for chaining."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax


def new_fig(figsize=(7.2, 4.2)):
    """One figure, one axis, pre-styled."""
    fig, ax = plt.subplots(figsize=figsize)
    style_axes(ax)
    return fig, ax

## 2. The state ladder — the C / L1 / O measurement view in state space (D43)

WS5's ablation is the same C / L1 / O ladder as the rest of the study, expressed in **state space** rather
than in features. Three nested tabular designs:

| design | state | ≈ view | states (non-terminal + 4 terminals) |
|---|---|---|---|
| `count` | `(balls, strikes)` | **C** | 12 + 4 = **16** |
| `count_prev` | `+` previous family (9, incl. `NONE`) | **L1** | 108 + 4 = **112** |
| `count_prev_trigger` | `+` a velo-gap **trigger flag** (2) | **O-lite** | 216 + 4 = **220** |

Every design shares four **absorbing terminals** — `walk`, `strikeout`, `hbp`, `in_play_end` — that end
the plate appearance. The plate appearance **is** the episode; its undiscounted return `Σ_t R_t` is, by
SPEC §5, the total run-value change, which is exactly what the OPE estimators score.

We first load the world and split it (train `2021–2023`, evaluate on `2024` + locked `2025`), then run
the full pipeline once so every downstream exhibit reads one consistent report.

In [ ]:
def load_world(mode):
    """Return (decision_table, truth_meta) for the chosen world, matching run_ws5's world build."""
    if mode == "real":
        if not REAL_TABLE_PATH.exists():
            raise FileNotFoundError(
                f"real decision table not found at {REAL_TABLE_PATH}. Build it with "
                "`python -m pitchseq.build_table` (RUNBOOK Step 1) first."
            )
        return pd.read_parquet(REAL_TABLE_PATH, engine="pyarrow"), {"world": "real"}
    from pitchseq.decision_table import build_decision_table
    from pitchseq.synth import make_null_world, make_positive_world
    if mode == "synth_null":
        raw, truth = make_null_world(n_games=NB_N_GAMES, seed=WORLD_SEED, innings_per_game=6)
    elif mode == "synth_positive":
        raw, truth = make_positive_world(n_games=NB_N_GAMES, seed=WORLD_SEED, innings_per_game=6,
                                         effect_size=0.5, velo_gap_threshold=THRESHOLD)
    else:
        raise ValueError(f"unknown DATA_MODE {mode!r}")
    return build_decision_table(raw), truth


table, truth = load_world(DATA_MODE)
splits = make_splits(table, CONFIG)["primary"]
train = table.loc[splits["train"].to_numpy()].reset_index(drop=True)
val = table.loc[splits["val"].to_numpy()].reset_index(drop=True)
test = table.loc[splits["test"].to_numpy()].reset_index(drop=True)
eval_rows = pd.concat([val, test], ignore_index=True) if len(test) else val.copy()

print(f"world      : {truth.get('world', DATA_MODE)}")
print(f"total rows : {len(table):,}   columns: {len(table.columns)}")
print(f"train rows : {len(train):,}   (2021-2023 -- the MDP is estimated here)")
print(f"eval rows  : {len(eval_rows):,}   (validation + locked test -- the rows the OPE scores)")
if DATA_MODE == "synth_positive":
    print(f"planted    : {truth.get('mechanism')}  effect={truth.get('effect')}  "
          f"threshold={truth.get('threshold')}  empirical_whiff_lift={truth.get('empirical_whiff_lift')}")

### Run the full OPE-gated pipeline once (every downstream exhibit reads this report)

`run_ws5` estimates the three MDPs, plans them, runs the **behavior-recovery gate first** (SPEC §0.3 /
D37), softens the greedy target toward behavior along `π_α`, scores it three ways per (design, α), runs
the **paired FQE refit bootstrap**, computes the setup diagnostics, and emits the verdict. We call it
once and print its headline; the exhibits below inspect the returned report `R5`. (On a synthetic world
`run_ws5` builds and caches the world itself; the small budgets keep this quick.)

In [ ]:
def run_full_pipeline(mode):
    """Run run_ws5 for the chosen world; return the report dict."""
    tmp = tempfile.mkdtemp(prefix="ws5_nb_")
    if mode == "real":
        ws3 = str(WS3_DIR) if WS3_DIR.exists() else None
        return run_ws5(source=str(REAL_TABLE_PATH), ws3_dir=ws3, synth="off", out=tmp,
                       alphas=ALPHAS, n_boot=NB_N_BOOT, fqe_boot=NB_FQE_BOOT,
                       sim_episodes=NB_SIM_EPISODES, seed=WORLD_SEED, write_outputs=True)
    world = "positive" if mode == "synth_positive" else "null"
    return run_ws5(synth=world, out=tmp, n_games=NB_N_GAMES, alphas=ALPHAS, alpha_r=ALPHA_R,
                   n_boot=NB_N_BOOT, fqe_boot=NB_FQE_BOOT, sim_episodes=NB_SIM_EPISODES,
                   seed=WORLD_SEED, write_outputs=True)


R5 = run_full_pipeline(DATA_MODE)
print(_format_headline(R5))

**The trigger flag — leakage-safe by construction.** The flag for the *current* decision is

$$\mathrm{trigger}_t = \mathbf{1}\!\left[\,t \ge 3 \ \wedge\ \bigl|\,v^{\mathrm{exec}}_{t-1} - v^{\mathrm{exec}}_{t-2}\,\bigr| \ge \tau\,\right],\qquad \tau = 5\ \text{mph},$$

computed from the **execution speeds of the two prior pitches only** (a within-PA shift of 1 and 2), so no
current-pitch execution ever enters the state (SPEC §0). It is exactly the planted positive-world mechanism.
The flag is `0` for the first two pitches of every PA, so a triggered state can arise only from `t ≥ 3`.

Below we encode each design on the eval rows to show the state counts live, decode a couple of sample
states, and read the empirical trigger rate.

In [ ]:
# Encode each design on the eval rows (pedagogy: the id scheme, live).
ss_by_design = {d: encode_states(eval_rows, d, THRESHOLD) for d in DESIGNS}
print(f"{'design':<22}{'S':>5}{'non-term':>10}{'terminals':>11}")
for d in DESIGNS:
    ss = ss_by_design[d]
    print(f"{d:<22}{ss.n_states:>5}{ss.n_nonterminal:>10}{len(TERMINALS):>11}")

# Decode a few richest-design states so the (count | prev | trigger) scheme is concrete.
# id 16 = the start (0-0, no prev, trigger 0); ids 18/19 are the SAME count+prev with the trigger off/on.
ssr = ss_by_design[RICHEST_DESIGN]
print("\nsample richest-design state labels (id -> human-readable):")
for sid in [16, 18, 19, ssr.terminal_ids["strikeout"]]:
    print(f"  id {sid:>4}  ->  {ssr.describe(sid)}")

# The leakage-safe trigger rate on the eval rows (prior pitches only; 0 on the first two of each PA).
prev_fam, trig, has2 = prev_and_trigger(eval_rows, THRESHOLD)
elig = has2.sum()
print(f"\ntrigger flag: {int(trig.sum()):,} of {len(trig):,} rows fire "
      f"({trig.mean():.1%}); {int(elig):,} rows have two priors ({trig[has2].mean():.1%} of those fire).")

**How to read the state counts.** The three designs are strictly nested — `count` collapses
`count_prev` over the previous family, which collapses `count_prev_trigger` over the trigger flag — so a
richer design can only *refine* what a coarser one sees. That refinement is the whole ablation: the count
design **cannot** represent a setup, because the trigger it would create is not in its state; the trigger
design can. The price is sparsity: `16 → 112 → 220` states estimated from the *same* logged rows, so the
richer designs spread their counts thinner. The next cell shows that cost directly.

## What "tabular" buys, and what it costs

**Buys:** total transparency and *exact* planning. The transition kernel, the reward table, and the value
function are all small matrices you can read; policy iteration solves the values by a linear system with no
approximation (§3). **Costs:** sparsity. Every `(state, action)` cell must be estimated from the logged
counts, and the richer designs have far more cells than the coarser ones. We plot the per-`(s, a)` support
so the estimation cost is visible — the richer designs have longer thin-cell tails, which is *why* their
in-sample optimism is larger and their held-out gap is modest even when the model-based gap looks big.

In [ ]:
# Per-(s,a) transition support N(s,a) for each design (estimated on train; the mask is illustrative).
mdps = {d: estimate_mdp(train, d, alpha_r=ALPHA_R, threshold=THRESHOLD) for d in DESIGNS}

fig, axes = plt.subplots(1, 3, figsize=(11.4, 3.4))
for ax, d in zip(axes, DESIGNS):
    style_axes(ax)
    term = mdps[d].terminal_mask()
    sup = mdps[d].support[~term].ravel()
    sup = sup[sup > 0]                                  # observed (s,a) cells only
    ax.hist(np.log10(sup + 1.0), bins=24, color=DESIGN_COLORS[d], alpha=0.85)
    reach = int((mdps[d].support.sum(axis=1) > 0).sum())
    ax.set_title(f"{d}\nS={mdps[d].n_states}  reach={reach}", fontsize=10.5)
    ax.set_xlabel("log10(1 + support per (s,a))")
axes[0].set_ylabel("observed (s,a) cells")
fig.suptitle("Per-(s,a) transition support -- sparsity grows with the ladder (D43)", y=1.05, fontweight="bold")
plt.show()

print(f"{'design':<22}{'reach/S':>10}{'mean feas/state':>17}{'median N(s,a)':>15}")
for d in DESIGNS:
    m = mdps[d]; term = m.terminal_mask()
    reach = int((m.support.sum(axis=1) > 0).sum())
    mean_feas = float(m.feasible.sum(axis=1)[~term].mean())
    obs = m.support[~term].ravel(); obs = obs[obs > 0]
    print(f"{d:<22}{f'{reach}/{m.n_states}':>10}{mean_feas:>17.2f}{np.median(obs):>15.0f}")

## 3. Estimation & planning

**Transitions (Dirichlet-smoothed).** With counts `N(s,a,s')` over the per-source-state *reachable* set
`succ(s)` of size `K_s`,

$$\hat P(s'\mid s,a) = \frac{N(s,a,s') + \alpha_t}{N(s,a) + \alpha_t\,K_s},\qquad s' \in succ(s),$$

zero elsewhere (`α_t = 1`). The dominant mass is the observed `N(s,a,s')`, which carry the exact next
previous family (`prev' = a`) and the velo-gap `trigger'`, so the **setup mechanism is preserved**; `α_t`
only regularises thin cells. A never-taken `(s,a)` backs off to the state's action-marginal transition.

**Rewards (two-level shrinkage).** The cell mean is shrunk toward the state mean, which is shrunk toward
the global mean `r̄`:

$$\hat r(s) = \frac{\sum_{i:s_i=s} R_i + \alpha_r\,\bar r}{n_s + \alpha_r},\qquad \hat R(s,a) = \frac{\sum_{i:(s_i,a_i)=(s,a)} R_i + \alpha_r\,\hat r(s)}{n_{s,a} + \alpha_r}.$$

Thin cells fall back smoothly to the state mean, then the global mean (no NaNs); terminals carry `R = 0`.

**Planning (undiscounted policy iteration, γ = 1).** Every PA terminates with probability one, so the
undiscounted values are finite and exact: policy evaluation solves `(I − P_π) V = R_π` on the non-terminal
block (terminals pinned at `V = 0`), and improvement takes the feasible arg-max, alternating until the
policy is stable. `γ = 1` is *correct*, not convenient — SPEC §5's return is the run-value change, and
discounting would distort it (a strike-three worth less than a strike-one).

The cell below verifies the smoothed kernel is a proper distribution, confirms policy iteration converged,
and reads the greedy value.

In [ ]:
d = RICHEST_DESIGN
mdp = mdps[d]
# Transitions are a proper distribution over reachable next-states, per (s, a).
row_sums = mdp.P.sum(axis=2)
print(f"[{d}] transition rows sum to 1: min={row_sums.min():.6f}  max={row_sums.max():.6f}")

# Plan with undiscounted policy iteration (gamma = 1).
policy_idx, Q, V = policy_iteration(mdp.P, mdp.R, mdp.feasible)
term = mdp.terminal_mask()
print(f"[{d}] policy iteration converged: |policy|={policy_idx.shape[0]} states, "
      f"V(start)={float(mdp.start_dist @ V):+.4f}")
print(f"[{d}] reward table: terminals R=0 -> {np.allclose(mdp.R[term], 0.0)};  "
      f"R range on non-terminals [{mdp.R[~term].min():+.3f}, {mdp.R[~term].max():+.3f}]")

# The start distribution is (essentially) a point mass -- every PA opens 0-0, no prev, trigger 0.
nz = int(np.count_nonzero(mdp.start_dist))
s0 = int(np.argmax(mdp.start_dist))
print(f"[{d}] start_dist nonzero entries: {nz}  (start state id {s0} = '{mdp.state_space.describe(s0)}')")

**The transparent simulator (D42) and the softening.** The simulator rolls the policy forward in the
estimated MDP — draw `s_0`, act `a ~ π[s]`, accrue `R[s,a]`, transition `s' ~ P[s,a]`, stop at a terminal —
and its Monte-Carlo return converges to the exact `Σ_s start_dist[s] V(s)`. That self-consistency is the
D42 validation of the model-based value; the pipeline already ran it, so we read it from `R5` per design.

The **primary** softening is the SPEC §9 mixture `π_α = (1−α)μ + απ_greedy` (built at the row level via
`ope.pi_alpha`); a temperature-softened Boltzmann variant is reported alongside as a secondary. `α = 0` is
behavior; `α = 1` is the pure greedy target. We dial one state to show the mixture.

In [ ]:
# D42 simulator self-consistency: the Monte-Carlo return reproduces the analytic model-based value.
print(f"{'design':<22}{'MB(greedy)':>12}{'sim(greedy)':>13}{'|diff|':>9}")
for dd in DESIGNS:
    ss = R5["state_summary"][dd]
    mb, sim = ss.get("model_based_greedy"), ss.get("simulated_greedy")
    if mb is not None and sim is not None:
        print(f"{dd:<22}{mb:>+12.4f}{sim:>+13.4f}{abs(mb - sim):>9.4f}")

# The pi_alpha mixture on one reachable trigger state (per-STATE view, for the exact model-based value).
mu_state = estimate_behavior_policy(train, encode_states(train, d, THRESHOLD))
greedy_state = greedy_policy_matrix(policy_idx, mdp.n_actions)
s = 19  # decode: (0-1 | prev=FF | trig=1) -- a triggered state, for illustration (label printed live)
print(f"\nstate {s} = '{mdp.state_space.describe(s)}'   pi_alpha(a|s) dialing behavior -> greedy:")
print(f"  {'family':<6}" + "".join(f"a={a:<5}" for a in ALPHAS))
for j, fam in enumerate(FAMILIES):
    if mdp.feasible[s, j] or greedy_state[s, j] > 0:
        row = [(1 - a) * mu_state[s, j] + a * greedy_state[s, j] for a in ALPHAS]
        print(f"  {fam:<6}" + "".join(f"{v:<7.3f}" for v in row))

## 4. The setup representation exhibit (D43) — does the model *value* creating a trigger?

The trigger flag makes a setup *representable*; the question is whether policy iteration actually **values**
it. `setup_diagnostics` reads two things off the richest MDP's optimal policy:

1. **Setup Q-gap.** For each reachable state, split the feasible actions into **trigger-creating** (velo
   band `|v(a) − v(prev)| ≥ τ`, so choosing `a` drives the *next* state's trigger to `1`) and non-creating,
   and report `max_creating Q(s,a) − max_non-creating Q(s,a)` — the extra value the MDP assigns to *setting
   up* a trigger. A positive mean is the model valuing the setup; `~0` means the trigger is inert.
2. **Optimal-action change.** The share of trigger-design states whose optimal action differs from their
   `count_prev` parent's (drop the trigger flag) — what the trigger knowledge *changed* in the policy.

**Completed validation (state these as done — the committed WS5a run).**

| diagnostic | positive world | null world |
|---|---|---|
| setup Q-gap mean (create vs not) | **+0.0087** (median +0.0079) | +0.0020 (median −0.0006) |
| setup Q-gap positive fraction | **0.68** (n = 140) | 0.49 (n = 140) |
| optimal-action change vs `count_prev` | **48.2%** of 141 states | 42.6% of 141 states |

The positive world's Q-gap is cleanly positive and its policy reshuffles nearly half the trigger states;
the null world's Q-gap sits at zero and its reshuffling is at the noise floor. The diagnostics **separate
the worlds** — the representation is doing real work where there is a real effect.

In [ ]:
# The live setup diagnostics from the pipeline report (computed on the richest MDP with the real mask).
diag = R5.get("setup_diagnostics", {})
if diag:
    print("setup diagnostics (count_prev_trigger MDP):")
    print(f"  setup Q-gap (create vs not): mean={diag['setup_gap_mean']:+.4f}  "
          f"median={diag['setup_gap_median']:+.4f}  pos-frac={diag['setup_gap_positive_frac']:.2f}  "
          f"(n={diag['n_states_with_gap']})")
    print(f"  optimal-action change vs count_prev: {diag['optimal_action_change_frac']:.1%} of "
          f"{diag['n_states_compared']} states  (mean trigger-creating actions/state="
          f"{diag['mean_creating_actions']:.2f})")

# Recompute the per-state Q-gap distribution directly, to plot it (create vs non-create max-Q per state).
mdp_t, mdp_p = mdps[RICHEST_DESIGN], mdps["count_prev"]
_pol_t, Q_t, _ = policy_iteration(mdp_t.P, mdp_t.R, mdp_t.feasible)
bands = mdp_t.velo_bands
reach = (mdp_t.support.sum(axis=1) > 0) & (~mdp_t.terminal_mask())
gaps = []
for s in np.flatnonzero(reach):
    cid_p, _tr = divmod(int(s), 2); _c, pidx = divmod(cid_p, N_PREV)
    feas = np.flatnonzero(mdp_t.feasible[s])
    if feas.size == 0 or pidx >= len(FAMILIES) or not np.isfinite(bands[pidx]):
        continue
    creating = np.isfinite(np.abs(bands[feas] - bands[pidx])) & (np.abs(bands[feas] - bands[pidx]) >= THRESHOLD)
    if creating.any() and (~creating).any():
        gaps.append(float(Q_t[s, feas[creating]].max() - Q_t[s, feas[~creating]].max()))
gaps = np.asarray(gaps)
fig, ax = new_fig(figsize=(7.2, 3.8))
ax.hist(gaps, bins=22, color=DESIGN_COLORS[RICHEST_DESIGN], alpha=0.85)
ax.axvline(0.0, color=REF_COLOR, lw=1.2, ls="--")
ax.axvline(gaps.mean() if gaps.size else 0.0, color=OKABE_ITO["orange"], lw=1.6,
           label=f"mean {gaps.mean():+.4f}" if gaps.size else "n/a")
ax.set_title("Setup Q-gap: value of a trigger-creating action vs the best non-creating one")
ax.set_xlabel("max_creating Q(s,a) - max_non-creating Q(s,a)"); ax.set_ylabel("states"); ax.legend()
plt.show()

### The representability proof — the constructed-world unit test, walked

The diagnostics *measure* the model valuing a setup; the unit test
`test_ws5_model.test_setup_representable_trigger_design_sets_up_and_wins` **proves** the machinery can cash
one, in a world small enough to check by hand. Its logic:

- Every PA is **three pitches**. Pitch 1 is `FF` (fast, 92 mph). Pitch 2 is a *choice*: `FF` (92) or `CU`
  (slow, 78). Pitch 3 is `FF`, ending in play.
- Choosing `CU` at pitch 2 makes `|velo_2 − velo_1| = 14 ≥ 5`, so **pitch 3's trigger fires** and its
  reward is a `boost` (here `0.5`); choosing `FF` leaves the trigger off and pitch 3 earns `0`. `CU` itself
  carries a small immediate **cost** (`−0.05`), so a *myopic* or *count* policy — which sees only this
  pitch's reward — avoids it.
- The **count** design's pitch-2 state is just `(0-1)`: it cannot see the trigger it would create, so its
  greedy policy takes `FF` and never pays the cost. The **trigger** design's pitch-2 state is
  `(0-1 | prev=FF | trig=0)`: it *can* see that `CU` drives pitch 3's trigger, so its greedy policy **takes
  `CU`** — it sets up.

The test asserts the trigger design's start value exceeds the count design's by a clear margin, and that
the transparent simulator reproduces the advantage. We rebuild the world inline and show it live.

In [ ]:
# Rebuild the constructed setup world inline (faithful to the unit test) and reproduce the proof.
def _setup_world(n_pa=120, boost=0.5, x_cost=0.05):
    rows = []
    for i in range(n_pa):
        take_cu = (i % 2 == 0)
        p2_fam, p2_velo, p2_R = ("CU", 78.0, -x_cost) if take_cu else ("FF", 92.0, 0.0)
        p3_trig = abs(p2_velo - 92.0) >= 5.0
        rows += [
            dict(pa_id=i, pitch_number=1, balls=0, strikes=0, family="FF", exec_release_speed=92.0,
                 outcome1="called_strike", R=0.0),
            dict(pa_id=i, pitch_number=2, balls=0, strikes=1, family=p2_fam, exec_release_speed=p2_velo,
                 outcome1="called_strike", R=p2_R),
            dict(pa_id=i, pitch_number=3, balls=0, strikes=2, family="FF", exec_release_speed=92.0,
                 outcome1="in_play", R=(boost if p3_trig else 0.0)),
        ]
    return pd.DataFrame(rows)

sw = _setup_world()
mdp_c = estimate_mdp(sw, "count", alpha_t=0.5, alpha_r=0.5)
mdp_tr = estimate_mdp(sw, "count_prev_trigger", alpha_t=0.5, alpha_r=0.5)
pol_c, _Qc, Vc = policy_iteration(mdp_c.P, mdp_c.R, mdp_c.feasible)
pol_t, _Qt, Vt = policy_iteration(mdp_tr.P, mdp_tr.R, mdp_tr.feasible)

s_count = 1                                   # (0-1) count-design state at pitch 2
s_trig = (1 * N_PREV + FIDX["FF"]) * 2 + 0    # (0-1 | prev=FF | trig=0) trigger-design state at pitch 2
v_c, v_t = float(mdp_c.start_dist @ Vc), float(mdp_tr.start_dist @ Vt)
print(f"pitch-2 optimal action -- count design  : {FAMILIES[pol_c[s_count]]}  (avoids CU's cost)")
print(f"pitch-2 optimal action -- trigger design: {FAMILIES[pol_t[s_trig]]}  (takes CU: the SETUP)")
print(f"start value             -- count / trigger: {v_c:+.4f} / {v_t:+.4f}   (trigger - count = {v_t - v_c:+.4f})")
sim_t, _ = simulate(mdp_tr.P, mdp_tr.R, greedy_policy_matrix(pol_t, mdp_tr.n_actions),
                    5000, np.random.default_rng(1), mdp_tr.start_dist)
sim_c, _ = simulate(mdp_c.P, mdp_c.R, greedy_policy_matrix(pol_c, mdp_c.n_actions),
                    5000, np.random.default_rng(1), mdp_c.start_dist)
print(f"simulator agrees        -- count / trigger: {sim_c:+.4f} / {sim_t:+.4f}")
ok = (FAMILIES[pol_t[s_trig]] == "CU") and (FAMILIES[pol_c[s_count]] == "FF") and (v_t > v_c + 0.1)
print(f"\nPROOF: trigger design sets up (CU) and wins, count design does not -> {'PASS' if ok else 'CHECK'}")

## 5. The D42 cross-check — three lenses on the *same* softened policy

WS5 is the study's OPE cross-validator. For each `(design, α)` the pipeline scores the **same** softened
policy `π_α` three ways:

- **model-based** — the exact value in the estimated MDP (`start_dist · V_π`); a point, half-width `0`;
- **step-wise DR** — the held-out doubly-robust value, with a cluster-bootstrap CI of its per-episode
  contributions;
- **FQE** — the held-out fitted-Q value, with the **refit** cluster-bootstrap CI (§6).

A pair **diverges** iff the absolute difference exceeds the wider of the two 95% CI half-widths (decision
D24, on the *real* CIs). Divergence is **reported, never silently averaged** — it is a *misspecification*
diagnostic, and it is exactly why the gate leans on the held-out FQE gap, not the in-sample model value.

**Completed validation (state these as done).** At `α = 0` the three lenses are **`CONSISTENT`** on every
design (the MDP's behavior value matches the held-out behavior value). As `α` rises and the design richens,
they **`DIVERGE`**: the estimated MDP's optimism at thin cells pulls the model-based value above the
held-out CIs. The sharpest verified example — richest design, `α = 1`: model-based `+0.0798` vs FQE
`+0.0459`, `|Δ| = 0.034` against a tolerance of `0.017` → **`DIVERGES`**.

In [ ]:
# The per-(design, alpha) D42 agreement grid, read from the report.
print(f"{'design':<22}{'alpha':>6}{'MB':>9}{'stepDR':>9}{'FQE':>9}{'ESS%':>7}   D42")
for dd in DESIGNS:
    for a in ALPHAS:
        cell = R5["ladder"][dd][a]
        print(f"{dd:<22}{a:>6.2f}{cell['model_based']:>+9.4f}{cell['stepwise_DR']['value']:>+9.4f}"
              f"{cell['FQE']['value']:>+9.4f}{cell['ess_frac']:>7.1%}   {cell['d42']['verdict']}")

# Zoom the sharpest divergence: richest design at the top alpha, the pairwise abs_diff vs tolerance.
top = max(ALPHAS)
d42 = R5["ladder"][RICHEST_DESIGN][top]["d42"]
print(f"\n[{RICHEST_DESIGN} @ alpha={top:.2f}] D42 = {d42['verdict']}")
for pair, info in d42["pairs"].items():
    tol = info.get("tolerance")
    print(f"  {pair:<24} abs_diff={info['abs_diff']:.4f}"
          + (f"  tol={tol:.4f}  diverges={info['diverges']}" if tol is not None else "  (no usable CI)"))

### The greedy-optimism exhibit — a *separate, labelled* figure (not a verdict input)

In-sample greedy values are **optimistic**: policy iteration selects the arg-max action per state, and the
max of noisy cell estimates is biased upward (max-selection bias), sharper on the sparser designs. This is
the D42 phenomenon at its most extreme, and we show it as its own exhibit so it is never mistaken for the
verdict. The bar is `model-based greedy − its own held-out FQE value` per design; a positive bar is the
optimism, and it grows with the ladder because the richer designs are estimated from thinner cells.

**Completed validation (positive world):** optimism `+0.0140` (count), `+0.0360` (count_prev), `+0.0339`
(count_prev_trigger) — the coarse design is the *least* optimistic. This is the tabular-support-honesty
exhibit, and it is the reason the held-out FQE gap, not the in-sample model value, is the gate (§6).

In [ ]:
# Greedy-optimism per design: in-sample model-based greedy minus its own held-out FQE value at alpha=1.
labels, mb_g, fqe_g, opt = [], [], [], []
for dd in DESIGNS:
    ss = R5["state_summary"][dd]
    fg = R5["ladder"][dd][max(ALPHAS)]["FQE"]["value"]
    mg = ss.get("model_based_greedy")
    labels.append(DESIGN_LABEL[dd]); mb_g.append(mg); fqe_g.append(fg); opt.append(mg - fg)

x = np.arange(len(DESIGNS)); w = 0.36
fig, ax = new_fig(figsize=(7.6, 4.0))
ax.bar(x - w / 2, mb_g, w, label="model-based greedy (in-sample)", color=OKABE_ITO["sky_blue"])
ax.bar(x + w / 2, fqe_g, w, label="FQE @ alpha=1 (held-out)", color=OKABE_ITO["orange"])
for xi, o in zip(x, opt):
    ax.annotate(f"optimism\n{o:+.4f}", (xi, max(mb_g) * 1.02), ha="center", fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels([l.replace(" (", "\n(") for l in labels], fontsize=9)
ax.set_ylabel("run value"); ax.set_title("Greedy-optimism exhibit (max-selection bias; NOT a verdict input)")
ax.legend(fontsize=9)
plt.show()

## 6. The honest gate — the D44 story, told straight

This is the chapter's methodological heart. The first WS5a build shipped a `SETUP_EXPLOITED` verdict. It
was wrong — not in arithmetic, but in **confidence** — and catching it is the finding.

**The defect (a structurally degenerate CI).** FQE's per-episode contribution is the initial-state value

$$V(s_0) = \sum_a \pi(a\mid s_0)\, \hat q(s_0, a),$$

and in this domain **every plate appearance starts in the identical state** — `0-0` count, no previous
pitch, trigger `0`. So the per-episode contribution array is a **constant**: every episode contributes the
same number. A cluster-resampling bootstrap of a constant array is degenerate — every resample has the same
mean, and the "CI" collapses to a single point `[x, x]`. The first gate rested on that fake zero-width
interval, which made a directional gap look *certified*. We show the degeneracy live.

In [ ]:
# D44, live: FQE's per-episode contribution is V(s0); every PA starts in the SAME state -> constant.
ev = eval_rows.sort_values(["pa_id", "pitch_number"], kind="stable").reset_index(drop=True)
ssr = encode_states(ev, RICHEST_DESIGN, THRESHOLD)
mdp_r = mdps[RICHEST_DESIGN]
pol_r, _Q, _V = policy_iteration(mdp_r.P, mdp_r.R, mdp_r.feasible)
target = policy_to_row_probs(greedy_policy_matrix(pol_r, mdp_r.n_actions), ssr.state_ids)

rew = pd.to_numeric(ev["R"], errors="coerce").to_numpy()
act = np.array([FIDX.get(str(f), XX_INDEX) for f in ev["family"]], dtype=np.int64)
keep = np.isfinite(rew)                                    # synthetic worlds are all-finite; guard anyway
logged = pd.DataFrame({"state": ssr.state_ids[keep], "action": act[keep], "reward": rew[keep],
                       "pa_id": ev["pa_id"].to_numpy()[keep],
                       "step": (ev["pitch_number"].to_numpy()[keep] - 1).astype(np.int64)})
fqe_val, fqe_contrib = ope.fqe(logged, target[keep], len(FAMILIES), state_col="state", action_col="action",
                               reward_col="reward", episode_id=logged["pa_id"].to_numpy(),
                               step=logged["step"].to_numpy(),
                               regressor_factory=onehot_tabular_regressor(len(FAMILIES)))
print(f"FQE point value            : {fqe_val:+.5f}")
print(f"per-episode contributions  : {len(fqe_contrib):,} episodes, "
      f"unique values = {np.unique(np.round(fqe_contrib, 8)).size}, spread (ptp) = {np.ptp(fqe_contrib):.2e}")
print("=> the contribution array is CONSTANT: a resampling bootstrap of it is structurally degenerate")
print("   (its CI would collapse to a point) -- exactly the D44 defect. The fix is a REFIT bootstrap.")

**The fix — the paired FQE refit cluster bootstrap.** The randomness that a real CI must capture is
**estimation** uncertainty: which held-out episodes we happened to log, and hence the *fitted Q*, not the
start state. So instead of resampling the constant contributions, each of `--fqe-boot` replicates resamples
**pitcher-game clusters of episodes** with replacement and **refits FQE from scratch** on the resampled
episodes — once per design and α, using the *same* resample for every arm so the design *gaps* are
**paired** (the shared episode-sampling noise differences away). The refits use `onehot_tabular_regressor`,
numerically identical to `ope.tabular_regressor` but orders of magnitude faster — which is what makes
hundreds of refits feasible. Wherever a CI is still structurally degenerate (e.g. the step-wise-DR gap at
`α = 0`, whose paired contributions are identically zero) it prints **`n/a (constant contributions)`**,
never a fake interval.

**Cost honesty.** At full synthetic scale the committed run is **200 replicates × 3 designs × 5 α = 3,000
FQE refits at ≈ 1.1 s/fit ≈ 57 minutes**, inside a demo of `~3,850 s` at peak `~1.2 GB`. On real data
(`~1.5M` held-out rows) lower `--fqe-boot` to 50–100 if slow — it changes only the CI resolution, never the
point estimates.

The cell reads the refit cost and the corrected triple-condition gate from the report.

In [ ]:
fr = R5.get("fqe_refit", {})
print(f"FQE refit bootstrap: {fr.get('n_boot')} replicates, {fr.get('n_fits')} refits, "
      f"{fr.get('seconds', float('nan')):.1f}s total, {fr.get('seconds_per_fit', float('nan')):.3f}s/fit "
      f"(committed run: 200 reps, 3000 refits, ~1.1s/fit, ~57 min)")

print(f"\nSETUP GAP vs the D40 myopic ceiling (+{R5['ceiling']:.3f}); trigger-count, FQE CI = paired refit:")
print("  gate: FQE lower-95 > ceiling AND stepDR gap > 0 AND model-based gap > 0, at the SAME alpha")
for a in ALPHAS:
    g = R5["gaps"].get(a, {}).get("trigger_minus_count")
    if not g:
        continue
    f, s, mb = g["FQE"], g["stepwise_DR"], g["model_based"]
    fci = "n/a" if f.get("degenerate") else f"[{f['ci95'][0]:+.4f},{f['ci95'][1]:+.4f}]"
    sci = "n/a" if s.get("degenerate") else f"[{s['ci95'][0]:+.4f},{s['ci95'][1]:+.4f}]"
    clears = (not f.get("degenerate")) and np.isfinite(f["lower_95"]) and f["lower_95"] > R5["ceiling"] \
        and s["value"] > 0 and mb > 0
    print(f"  alpha={a:>4.2f}  FQE={f['value']:+.4f} {fci} lo95={f['lower_95']:+.4f}"
          f"  | stepDR={s['value']:+.4f} {sci}  | MB={mb:+.4f}"
          + ("   <== CLEARS (all 3)" if clears else ""))

**The general lesson — in sequential OPE, the variance *is* the claim.** WS4 learned it at the myopic
level: its point estimate was directionally right, but the OPE noise floor swallowed the `0.003` edge. WS5
learns the same lesson one rung up, and more sharply, because the degeneracy was *silent* — the naive
bootstrap did not error, it returned a confident lie. The discipline that falls out is a **binding reading
rule**, stated before any branch is read:

> **Gate on a real interval, never on a point.** A prescriptive claim requires a **non-degenerate** CI
> whose one-sided lower bound clears the target. A degenerate CI (constant contributions) is printed
> `n/a`, never rendered as an interval, and can never fire a positive verdict. Point estimates and
> in-sample model values are *evidence*, not *certification*.

This is WS5's analogue of WS4's myopic-ceiling rule and WS1's D21 Δ_order rule: the honest criterion is
written down *before* the numbers, so a degenerate interval can never be quietly promoted to a win.

## 7. The verdict — `SETUP_INCONCLUSIVE`, which is decision D39 working

On the positive world the corrected gate is **not met at synthetic scale**, and the honest verdict is
`SETUP_INCONCLUSIVE`. This is not a failure — it is D39 (pre-written honest negatives are first-class
results) doing its job. The evidence story the headline prints:

- **All three lenses are directionally positive at `α = 1`:** FQE gap **`+0.0079`** (CI `[−0.0081, +0.0218]`,
  one-sided lower bound `≈ −0.004`), step-wise-DR gap **`+0.0107`** (CI `[−0.061, +0.077]`), model-based gap
  **`+0.0277`** — directional agreement, `True`.
- **The ladder discriminates the worlds:** the positive world's held-out FQE trigger-count gap (`+0.0079`)
  is positive and clearly above the null world's (`−0.0106`), and the setup diagnostics separate them
  (Q-gap mean `+0.0087` vs `+0.0020`; optimal-action change `48.2%` vs `42.6%`).
- **None of it clears the `+0.003` ceiling with a real CI:** the FQE gap's lower bound is negative, and the
  step-wise-DR gap's CI is `~±0.07` wide — the sequential-OPE variance (D44) is the binding constraint.

So the fixture's effect is **real, representable, and statistically uncertifiable at this scale**. The
machinery *provably* cashes a setup (§4); certifying it *in the wild* is a **data-scale** question the
Phase-2 full-data run answers.

In [ ]:
v = R5.get("verdict", {})
ev = v.get("evidence", {})
print("=" * 68)
print(f" WS5 VERDICT : {v.get('verdict')}")
print("=" * 68)
if ev:
    print(f" at alpha={ev.get('alpha')}:  FQE gap={ev.get('fqe_value'):+.4f} (lo95={ev.get('fqe_lower_95'):+.4f})"
          f"  stepDR gap={ev.get('stepdr_value'):+.4f}  MB gap={ev.get('model_based'):+.4f}")
    print(f" directional agreement (all three > 0) : {ev.get('directional_agreement')}")
print(f" setup Q-gap mean={v.get('setup_gap_mean')}  pos-frac={v.get('setup_gap_positive_frac')}")
print(f" optimal-action change on trigger states : {v.get('optimal_action_change_frac')}")
print("=" * 68)

# Back-of-envelope: the n required to certify a +0.008 gap. CI half-width h ~ c / sqrt(n_clusters).
top = max(ALPHAS)
g = R5["gaps"][top]["trigger_minus_count"]["FQE"]
lo, hi = g["ci95"]
if np.isfinite(lo) and np.isfinite(hi):
    h = (hi - lo) / 2.0
    signal = 0.008                                   # the committed FQE gap point (~+0.0079)
    target_h = (signal - R5["ceiling"]) / 1.645      # half-width needed so lower_95 clears the ceiling
    factor = (h / target_h) ** 2 if target_h > 0 else float("inf")
    print(f"\n n-to-certify (back-of-envelope): current half-width h~{h:.4f}; to put lower_95 above the")
    print(f" +{R5['ceiling']:.3f} ceiling for a +{signal:.3f} gap needs h<~{target_h:.4f} -> ~{factor:.0f}x the")
    print(f" effective clusters (h ~ 1/sqrt(n)); full data is ~40x the fixture -- a Phase-2 question.")

**This is D39 working.** The pipeline never fabricated a clearance it could not defend, and it never
buried the positive evidence either. It printed the full story — the directional agreement, the world
discrimination, the setup diagnostics, the pointer to the constructed-world proof — and then said the honest
thing: *not yet, at this scale.* The `n`-to-certify arithmetic above (half-width shrinks like `1/√n`, so
resolving the gap needs several-fold more effective clusters, and the full data is `~40×` the fixture)
turns "inconclusive" into a concrete, falsifiable **Phase-2 prediction** rather than a shrug.

## 8. Results — branched interpretation (design-ladder × D42 × verdict)

The real-data result is read on three axes. Exactly one branch on each axis applies, and each is written to
stand alone once the numbers above are filled. The **design-ladder** axis is WS5's actual question (does a
richer state buy sequential prescription value?); the **D42** axis qualifies how much to trust the value
(do the three lenses agree?); the **verdict** axis is the headline. The binding reading rule from §6 governs
all three: *gate on a real CI, never on a point; a degenerate CI is `n/a`, never an interval.*

The code cell inspects the computed report and prints which branch fired on each axis; the markdown that
follows holds the pre-written interpretation for every branch.

In [ ]:
# --- Design-ladder axis (S): the held-out trigger-count FQE gap at the top alpha, gated on a REAL CI ---
top = max(ALPHAS)
tc = R5["gaps"][top].get("trigger_minus_count", {})
fqe_gap = tc.get("FQE", {})
lo95 = fqe_gap.get("lower_95", float("nan"))
val = fqe_gap.get("value", float("nan"))
hi = fqe_gap.get("ci95", [float("nan"), float("nan")])[1]
degen = fqe_gap.get("degenerate", True)
if (not degen) and np.isfinite(lo95) and lo95 > R5["ceiling"]:
    s_branch = "S+"                                   # clears the ceiling with a real CI
elif np.isfinite(hi) and hi < 0:
    s_branch = "S-"                                   # richer design significantly LOSES
else:
    s_branch = "S0"                                   # directional but uncertifiable (or null-inert)

# --- D42 axis (A): do the three lenses agree anywhere they are assessed? ---
verdicts = [R5["ladder"][d][a]["d42"]["verdict"] for d in DESIGNS for a in ALPHAS]
a_branch = "A-" if any(x == "DIVERGES" for x in verdicts) else "A+"

# --- Verdict axis ---
verdict = R5.get("verdict", {}).get("verdict")

print("=" * 68)
print(" WS5 RESULTS -- branch selector")
print("=" * 68)
print(f" world / mode          : {R5.get('world')}")
print(f" trigger-count FQE gap @ alpha={top:.2f}: {val:+.4f}  (lo95 {lo95:+.4f}, ceiling +{R5['ceiling']:.3f})")
print(f" D42 cells DIVERGES     : {sum(x == 'DIVERGES' for x in verdicts)}/{len(verdicts)}")
print("-" * 68)
print(f" DESIGN-LADDER axis : {s_branch}   (the real question -- does the trigger state buy value?)")
print(f" D42 AGREEMENT axis : {a_branch}   (trust the value only where the lenses agree)")
print(f" VERDICT axis       : {verdict}")
print("=" * 68)
print(" -> read the matching branch write-ups in the markdown below.")

### Design-ladder axis (S+ / S0 / S−)

**S+ — the trigger/prev designs beat `count` with real-CI clearance.** At some α the refit-bootstrap FQE
trigger-count gap's one-sided lower bound clears the `+0.003` ceiling — **sequential prescription value
found**, a setup the myopic bandit provably could not cash. This is the strong result and a strong claim:
cross-check it against **WS3's H-branch** (was the order even predictive out of sample?) and **WS4's
P-branch** (a myopic P0/`SEQ_INCONCLUSIVE_MYOPIC` there, exceeded here, is exactly the D40 handoff working).
Confirm the step-wise-DR and model-based gaps are directionally positive at the same α, and that the D42
column is not `DIVERGES` at that cell. If it survives, the burden passes to WS7's full offline-RL battery.

**S0 — directional but uncertifiable (the synth-validated pattern).** The gap is directionally positive in
all three lenses and above the null world's baseline, but its refit-bootstrap lower bound does not clear the
ceiling — the fixture's own `SETUP_INCONCLUSIVE`. Report the *evidence story*, not a null: directional
agreement, the world-discriminating diagnostics, and the constructed-world proof that the machinery cashes
setups. Then state the `n`-to-certify back-of-envelope (half-width `~1/√n`; full data is `~40×` the fixture)
as a concrete Phase-2 prediction. This is the most anticipated real-data outcome at moderate scale.

**S− — the richer designs lose (sparsity / overfit).** The trigger-count gap's CI upper bound is below zero:
the richer state is worth *less* than `count` out of sample. Read it through the support histograms (§2) and
the greedy-optimism exhibit (§5) — the `216`-state design is estimated from far thinner cells than the
`16`-state one, so its extra resolution adds estimation variance without prescriptive signal (the state-space
echo of WS3's negative `Δ_matchup` and WS4's `O`-view overfitting). The honest reading is "no sequential
prescription edge **and** a real estimation cost of the richer state," and the policy handed downstream
should be built from the coarser design.

### D42 agreement axis (A+ / A−)

**A+ — the three lenses agree (trust the value).** Model-based, step-wise DR, and FQE sit inside each other's
95% CIs wherever both are assessable — the estimated MDP, the importance-weighted lens, and the fitted-Q
lens tell one story. The value is a coherent read, and the design-ladder gap can be taken at face value
(subject to its own CI). Expect this at low α, where the softened policy stays near behavior and the MDP's
in-sample value matches the held-out world.

**A− — `DIVERGES` (a misspecification diagnostic, not an averaging problem).** The estimated MDP's value of
the softened policy sits *outside* the held-out OPE's real CIs — reported per D42, never silently averaged.
On the synthetic worlds this grows with α and with design richness: it is the tabular state's **in-sample
optimism at thin cells**, and it is *informative*. It says the tabular Markov state is **not
Markov-sufficient** — the count-plus-trigger abstraction drops real structure that the held-out data feels,
so the model-based value overshoots. For **WS7** this is the explicit brief: a richer function class (a
learned representation) is the response to a `DIVERGES` here, and the same paired-refit gate philosophy
carries over. When A− fires, lean on the held-out FQE gap and its refit CI — never the in-sample model
value — for the verdict.

### Verdict axis (`SETUP_EXPLOITED` / `SETUP_INCONCLUSIVE` / `SEQ_NEUTRAL_MDP`), under D39

**`SETUP_EXPLOITED`.** At some α the triple condition holds — refit-FQE trigger-count lower-95 above the
ceiling **and** step-wise-DR gap directionally positive **and** model-based gap positive. The tabular MDP
**cashes a setup pitch** on held-out data beyond the myopic ceiling: the ladder's motivating contrast
delivered. Never fabricated — it fires only on a real, non-degenerate lower bound.

**`SETUP_INCONCLUSIVE` (D39 first-class, the fixture's verdict).** No α meets the triple condition on a
positive world. The effect is present, representable, and directionally agreed, but the sequential-OPE
variance cannot certify it at this scale. This is the honest negative WS4's `SEQ_INCONCLUSIVE_MYOPIC` has a
sequential twin — and the certification is deferred to Phase-2 data scale, with the `n`-to-certify arithmetic
as the falsifiable prediction.

**`SEQ_NEUTRAL_MDP` (the null world's expected result).** No α meets the triple condition and the design
ladder shows no held-out advantage — the trigger flag is inert, the richer designs are worth no more than
`count` up to overfitting cost. The setup diagnostics sit at their noise floor (Q-gap `~0`, optimal-action
change at the reshuffling baseline). Read it as "the machinery correctly finds nothing where there is
nothing," the null control passing.

#### Reading the grid

The honest headline is a triple `(S, A, verdict)`. The study's *most anticipated* real-data cell is
**S0 × A− × `SETUP_INCONCLUSIVE`** at moderate scale — a real setup effect present but below the
sequential-OPE floor, with the tabular state visibly imperfect (A−), motivating WS7. The *strongest* cell is
**S+ × A+ × `SETUP_EXPLOITED`**: a certified setup with agreeing lenses, to be cross-checked hard against
WS3/WS4 and handed to WS7. The *null* cell is **S0(inert) × (A±) × `SEQ_NEUTRAL_MDP`**. The *diagnostic*
cell is any **S−** (richer-design overfit — build from the coarser state).

## 9. Discussion and limitations

**The WS4 → WS5 → WS7 arc.** WS4 measured the myopic ceiling and handed WS5 a falsifiable target; WS5 proved
the setup is *representable* and cashable in principle, and bound its *certification* to the sequential-OPE
variance; WS7 inherits both — the same paired-refit gate philosophy and a richer function class to answer the
`DIVERGES` diagnostic. Each rung's honest negative is the next rung's motivating contrast.

**Limitations.**

1. **Tabular Markov-sufficiency.** The state is `(count, prev-family, trigger)` plus terminals. If the true
   dynamics depend on more (exact velocities, location, the batter), the tabular state is not Markov-sufficient
   and the model-based value is biased — which is *exactly* what a D42 `DIVERGES` reports. WS5 measures this
   honestly; it does not fix it (WS7's job).
2. **One engineered mechanism.** The trigger flag encodes *one* setup mechanism — a large ordered velo
   transition — because that is the one the fixture plants. Real data has unknown mechanisms; the MDP tests
   only *representable* ones, and a real setup that does not project onto the trigger is invisible here. The
   claim is never "this is the setup mechanism," only "this representable one is (not) cashable."
3. **Certification is variance-bound, not point-bound.** The binding constraint is the refit-bootstrap CI
   width, not the gap's sign. At synthetic scale it cannot clear the ceiling; whether it does at full scale is
   the open question, and the `n`-to-certify arithmetic (§7) is a back-of-envelope, not a guarantee.
4. **Refit-bootstrap cost on real data.** The default `--fqe-boot 200` is `~3,000` refits; at `~1.5M` held-out
   rows budget up to a few hours, and lower `--fqe-boot` to 50–100 if slow (RUNBOOK WS5.1) — it changes only
   the CI resolution, never the point estimates.
5. **Behavior model.** Without `--ws3-dir`, the OPE denominator is the state-conditional empirical behavior
   from train counts — coarser than WS3's contextual propensities. The behavior-recovery gate still guards
   it, but a richer `μ` (WS3) tightens the importance weights.
6. **The firewall.** WS5 tests whether *acting* on the estimated MDP beats behavior *within support*; it does
   not certify the trigger as *causal*. `SETUP_EXPLOITED` is evidence the ordered state yields a better
   *evaluable* policy, not proof that *changing* the sequence *causes* the gain — the finding-#3 claim only
   WS7's full battery can approach.

## 10. Reproducibility appendix

**Phase-2 commands (RUNBOOK WS5.1).** WS5 needs only the decision table (Step 1); `--ws3-dir` is optional
(reuses WS3's behavior propensities, decision D33; without it, the empirical fallback).

```powershell
conda activate statcast; cd ~\pitch-sequencing-research
# WS5.1 -- estimate the D43 tabular MDPs, plan, and evaluate the setup value (OPE-gated)
python workstreams/ws5_tabular_mdp/run_ws5.py --table data/processed/decision_table.parquet --ws3-dir results/ws3/ --out results/ws5/
```

The synthetic Phase-1 CI equivalents (no data, no dependency -- WS5 builds and caches the world itself):
`--synth null --out results/ws5_null/` and `--synth positive --out results/ws5_pos/`. `--alpha-t` /
`--alpha-r` are the transition / reward smoothing strengths; `--threshold` (default 5.0) echoes the velo-gap
trigger; `--n-boot` sets the step-wise-DR contribution bootstrap; **`--fqe-boot`** (default 200) sets the
FQE refit bootstrap — the resolving CI; **lower it to 50–100 on the real table if slow** (it changes only
the CI resolution, never the point estimates). `--force` rebuilds the cached synthetic table.

**Artifact inventory (written under `results/ws5/`).**

- `policy_<world>_<design>.parquet` — standard-schema `policy_prob` predictions per design (the pure greedy
  target; design → view map is `count`→`C`, `count_prev`→`L1`, `count_prev_trigger`→`O`; the exact design
  is in `model_id`).
- `ladder_<world>.csv` — the state-ladder overlay data (model-based, step-wise DR, FQE, ESS per design × α).
- `ws5_report_<world>.json` — the full report (gate, ladder, D42 cross-check, gaps vs the ceiling, setup
  diagnostics, verdict).
- `ws5_<world>.runmeta.json` — timing / peak RAM for the SPEC §7 Pareto plot.

**What to look at first**, in order: (1) the **gate** — the line after `rows` must read `behavior recovery:
PASS`; a `FAILED_GATE` stops the run before any target value. (2) the **SETUP GAP vs the D40 ceiling** block
— the *refit-bootstrap* FQE trigger-count gap and whether its lower-95 clears `+0.003`. (3) the **verdict**.
(4) the **D42 column** and the **greedy-optimism exhibit** — read the value only where the lenses agree.
The exact formulas live in `THEORY.md`; the exact code in `model.py` / `run_ws5.py`; the plain-English tour
in `SEAN-README.md`.

In [ ]:
import scipy, sklearn
print("python     :", platform.python_version())
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("scipy      :", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib :", matplotlib.__version__)
print("seed       :", SEED, " world_seed:", WORLD_SEED)
print("data mode  :", DATA_MODE)
print("designs    :", DESIGNS, " richest:", RICHEST_DESIGN)
print("alphas     :", ALPHAS, " threshold:", THRESHOLD, " ceiling(D40):", D40_MYOPIC_CEILING)
print("budgets    : n_games", NB_N_GAMES, " fqe_boot", NB_FQE_BOOT, " n_boot", NB_N_BOOT)